# SMS Spam Collection - preprocessing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/smsspam/00_preprocessing/smsspam_preprocessing.ipynb)

Stateless steps only: label mapping, removing identifiers and duplicates, marking invalid values
as missing, and (for large data) a fixed stratified sample. Everything that is *fitted*
(imputation, scaling, TF-IDF, resampling) happens later inside each training fold.

Writes `data/processed/smsspam.parquet` and the fixed 5x2 outer folds `data/splits/smsspam_outer.npz`.
Both are committed, so every experiment notebook (and your teammate) uses identical data and folds.
Raw data is expected in `data/raw/` (not committed; see `data/README.md`).

In [1]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


Environment OK: scikit-learn 1.9.1, imbalanced-learn 0.14.2, xgboost 3.4.1, lightgbm 4.7.0


## Load
UCI SMS Spam Collection: 5,574 English SMS messages labelled ham or spam (tab-separated).

In [2]:
import zipfile
import pandas as pd
from common import data, settings
from common.paths import RAW_DIR

with zipfile.ZipFile(RAW_DIR / "sms+spam+collection.zip") as z:
    lines = z.read("SMSSpamCollection").decode("utf-8", errors="replace").splitlines()
raw = pd.DataFrame([l.split("\t", 1) for l in lines if l.strip()], columns=["label", "text"])
print(raw.shape)
raw["label"].value_counts()

(5574, 2)


label
ham     4827
spam     747
Name: count, dtype: int64

## Clean (stateless steps only)
* Label: spam -> 1 (minority), ham -> 0.
* Remove exact duplicate messages **before** splitting. Otherwise the same message
  can sit in both a training and a test fold, which is leakage (reviewer R10).
* Remove messages that appear with both labels (none expected).

TF-IDF and SVD are **not** fitted here; they are fitted inside each training fold.

In [3]:
df = raw.assign(y=(raw["label"] == "spam").astype(int))[["text", "y"]]
print("duplicate messages:", df.duplicated().sum())
df = df.drop_duplicates()
conflict = df.duplicated("text", keep=False)
print("messages with both labels:", conflict.sum())
df = df[~conflict].reset_index(drop=True)
print(df.shape)
df["y"].value_counts()

duplicate messages: 403
messages with both labels: 0
(5171, 2)


y
0    4518
1     653
Name: count, dtype: int64

## Save processed data and the 5x2 outer folds

In [4]:
data.save_processed("smsspam", df)
data.make_outer_splits(df["y"], "smsspam")
data.summary("smsspam")

{'dataset': 'smsspam',
 'n': 5171,
 'd': 1,
 'n_minority': 653,
 'minority_pct': 12.628,
 'IR': 6.92}